# Migrate Existing Data To V3

This notebook copies existing saved data into the V3 file-saving structure without rerunning simulations.

The migration is copy-only. Your original V2 files stay unchanged.

Run this notebook from your HOOMD/MD conda kernel so `h5py`, `gsd`, and the project helpers are available.

In [ ]:
import importlib
from pathlib import Path

import pandas as pd

import md_Helpers.paths as paths
import md_Helpers.migration as migration

importlib.reload(paths)
importlib.reload(migration)

print("Project root:", paths.PROJECT_ROOT)
print("V3 lattices:", paths.SIMPLE_LATTICES_V3_ROOT)
print("V3 thermalized states:", paths.THERMALIZED_STATES_V3_ROOT)


## Migration Settings

Start with `LIMIT = 10` for a small smoke test. Use `LIMIT = None` for all matching files.

`migrate_cavitation_states` is off by default because cavitation has not become the main run pipeline yet. Turn it on later if you want to copy existing initial cavitation states too.

In [ ]:
MIGRATE_LATTICES = True
MIGRATE_THERMALIZED = True
MIGRATE_CAVITATION_STATES = False

# Use 10 first. Change to None after the plan looks right.
LIMIT = 10

# Keep False unless you intentionally want to replace existing V3 files.
OVERWRITE = False

# Keep True. This writes grouped V3 metadata into copied V3 HDF5 logs.
WRITE_V3_METADATA = True


## Dry Run

This cell does not copy anything. It shows exactly what would be copied and where it would go.

In [ ]:
dry_run_report = migration.migrate_v2_to_v3(
    dry_run=True,
    link_mode="copy",
    overwrite=OVERWRITE,
    migrate_lattices=MIGRATE_LATTICES,
    migrate_thermalized=MIGRATE_THERMALIZED,
    migrate_cavitation_states=MIGRATE_CAVITATION_STATES,
    write_v3_metadata=WRITE_V3_METADATA,
    limit=LIMIT,
)

print("Dry-run rows:", len(dry_run_report))

if len(dry_run_report) > 0:
    display(dry_run_report["object_kind"].value_counts(dropna=False))
    display(dry_run_report["action"].value_counts(dropna=False))
    display(dry_run_report.head(25))
else:
    display(dry_run_report)


## Full Dry Run

After the small dry run looks good, run this cell to preview the full migration. It still does not copy anything.

In [ ]:
full_dry_run_report = migration.migrate_v2_to_v3(
    dry_run=True,
    link_mode="copy",
    overwrite=OVERWRITE,
    migrate_lattices=MIGRATE_LATTICES,
    migrate_thermalized=MIGRATE_THERMALIZED,
    migrate_cavitation_states=MIGRATE_CAVITATION_STATES,
    write_v3_metadata=WRITE_V3_METADATA,
    limit=None,
)

print("Full dry-run rows:", len(full_dry_run_report))

if len(full_dry_run_report) > 0:
    display(full_dry_run_report["object_kind"].value_counts(dropna=False))
    display(full_dry_run_report["action"].value_counts(dropna=False))
    display(full_dry_run_report.head(25))


## Real Migration

This cell actually copies files. It is guarded by `RUN_REAL_MIGRATION = False` so you cannot run it accidentally.

When you are ready, set `RUN_REAL_MIGRATION = True` and run the cell.

In [ ]:
RUN_REAL_MIGRATION = False

if not RUN_REAL_MIGRATION:
    raise RuntimeError("Set RUN_REAL_MIGRATION = True only after reviewing the dry-run output.")

migration_report = migration.migrate_v2_to_v3(
    dry_run=False,
    link_mode="copy",
    overwrite=OVERWRITE,
    migrate_lattices=MIGRATE_LATTICES,
    migrate_thermalized=MIGRATE_THERMALIZED,
    migrate_cavitation_states=MIGRATE_CAVITATION_STATES,
    write_v3_metadata=WRITE_V3_METADATA,
    limit=None,
)

print("Migration rows:", len(migration_report))

if len(migration_report) > 0:
    display(migration_report["object_kind"].value_counts(dropna=False))
    display(migration_report["action"].value_counts(dropna=False))
    display(migration_report.head(25))

paths.MASTER_CSVS_V3_ROOT.mkdir(parents=True, exist_ok=True)
report_path = paths.MASTER_CSVS_V3_ROOT / "v3_migration_report.csv"
migration_report.to_csv(report_path, index=False)

print("Saved migration report:", report_path)


## Quick Verification

Run this after a real migration to check that representative destination files exist.

In [ ]:
report_to_check = globals().get("migration_report", None)

if report_to_check is None:
    print("No migration_report variable found yet. Run the real migration cell first.")
else:
    check_df = report_to_check.copy()

    if "destination_path" in check_df.columns:
        check_df["destination_exists"] = check_df["destination_path"].apply(
            lambda value: Path(value).exists() if pd.notna(value) else False
        )

        display(check_df["destination_exists"].value_counts(dropna=False))
        display(check_df.head(25))

    if "metadata_path" in check_df.columns:
        metadata_rows = check_df[check_df["metadata_path"].notna()].copy()
        metadata_rows["metadata_exists"] = metadata_rows["metadata_path"].apply(
            lambda value: Path(value).exists()
        )
        display(metadata_rows["metadata_exists"].value_counts(dropna=False))


In [12]:
from pathlib import Path
from md_Helpers import classification

DATA_ROOT = Path("/exp/e961/data/MDsims-data/pnichols")
THERMALIZED_V3_ROOT = DATA_ROOT / "Thermalized_States_v3"
REPORT_ROOT = DATA_ROOT / "Master_CSVs_v3"

# Dry run: shows what would be normalized
dry_report = classification.normalize_all_phase_separation_metadata_paths(
    root=THERMALIZED_V3_ROOT,
    dry_run=True,
    remove_legacy=True,
    overwrite=False,
)

print("Dry-run rows:", len(dry_report))
display(dry_report["status"].value_counts(dropna=False))
display(dry_report["methods_found"].value_counts(dropna=False))
display(dry_report.head(20))

# Set this to True only after the dry run looks right.
RUN_REAL_CLEANUP = True

if RUN_REAL_CLEANUP:
    cleanup_report = classification.normalize_all_phase_separation_metadata_paths(
        root=THERMALIZED_V3_ROOT,
        dry_run=False,
        remove_legacy=True,
        overwrite=False,
    )

    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    report_path = REPORT_ROOT / "v3_phase_metadata_cleanup_report.csv"
    cleanup_report.to_csv(report_path, index=False)

    print("Cleanup rows:", len(cleanup_report))
    print("Saved cleanup report:", report_path)
    display(cleanup_report["status"].value_counts(dropna=False))
    display(cleanup_report["methods_found"].value_counts(dropna=False))
    display(cleanup_report.head(20))
else:
    print("Dry run only. Set RUN_REAL_CLEANUP = True to actually modify files.")

Dry-run rows: 386


status
dry_run    386
Name: count, dtype: int64

methods_found
PE_drop,voxel    260
voxel            126
Name: count, dtype: int64

,status,log_path,methods_found,sources_found,removed_legacy
0,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
1,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
2,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
3,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
4,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
5,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
6,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
7,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
8,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False
9,dry_run,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",False


Cleanup rows: 386
Saved cleanup report: /exp/e961/data/MDsims-data/pnichols/Master_CSVs_v3/v3_phase_metadata_cleanup_report.csv


status
normalized    386
Name: count, dtype: int64

methods_found
PE_drop,voxel    260
voxel            126
Name: count, dtype: int64

,status,log_path,methods_found,sources_found,removed_legacy
0,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
1,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
2,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
3,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
4,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
5,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
6,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
7,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
8,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
9,normalized,/exp/e961/data/MDsims-data/pnichols/Thermalize...,"PE_drop,voxel","metadata/classification/PE_drop,metadata/class...",True
